# 📓 **Notebook 1 — Phases 1 & 2**

**Project:** [Medical Transcription Classfier and Semantic Search Engine Project](https://github.com/bahadirkoko/medical-transcription-classifier)

## **Medical Transcription Analysis — Phases 1 & 2**

This notebook covers exploratory analysis and statistical feature analysis of the
Medical Transcriptions dataset, preparing the groundwork for classification (Phase 3)
and semantic search (Phase 4).

## **Phase 1 — EDA & NLP Foundations**
Clean the data and understand the linguistic "fingerprint" of each medical specialty.
- **1.1 Cleaning & Features** — remove nulls/duplicates/short notes; compute length,
  word, and sentence counts; visualize class distribution and note lengths.
- **1.2 NLP Analysis** — preprocess text (lowercase, remove punctuation, drop stopwords);
  find top words, bigrams, and trigrams per specialty.

## **Phase 2 — Statistical Significance & Linguistic Importance**
Statistically measure which words and phrases are tied to each specialty.
- **2.1 Chi-Square** — tests which words are statistically "locked" to a specialty.
- **2.2 N-Grams + PMI** — finds meaningful medical phrases vs. random word pairings.
- **2.3 TF-IDF** — scores words by uniqueness, filtering out common "medical noise."


---

### 🗂️ **Project notebooks**
1. **Notebook 1** — Phases 1 & 2: Collection & Cleaning **(you are here)**
2. Notebook 2 — Phase 3: Exploratory Analysis
3. Notebook 3 — Phases 4 & 5: Feature Engineering & Modeling

In [326]:
#data libraries
import pandas as pd
import numpy as np

#visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

#text/NLP libraries
import re #punctuation removal
import nltk #cleaning, tokenization, stopword removal, n-grams
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.util import ngrams
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder
from collections import Counter #n-gram frequency counting

#ML work phase 1.2, vectorization and phase 2
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import chi2

#NLTK data downloads
nltk.download('punkt') #downloads the Punkt tokenizer models, which are used for sentence splitting and tokenization.
nltk.download('punkt_tab') #downloads the Punkt tokenizer models for tokenizing text into sentences and words, specifically for tab-separated text.
nltk.download('stopwords') #downloads the stopwords corpus, which contains a list of common stopwords in various languages that can be used for text preprocessing and filtering.


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/bahadirkocabas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/bahadirkocabas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bahadirkocabas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Phase 1.1 EDA

Before any cleaning, we load the raw dataset and inspect its structure —
dimensions, column quality (nulls and duplicates), and transcription length
distribution. This informs every cleaning decision that follows: we only
drop what we can justify from what we observe here.


In [346]:
df = pd.read_csv('../data/raw/medicaltranscriptionsamples.csv') #loading data from a relative path

print("Rows:", df.shape[0], "Columns:", df.shape[1]) #to get the number of rows and columns in the DataFrame
display(df.head())

df = df.drop(columns=['Unnamed: 0']) #dropping the unnamed column 

null_and_duplicated = pd.DataFrame({
    'null_%': (df.isnull().sum() / len(df) * 100).round(2), #getting the percentage of null values in each column, rounded to 2 decimal places, no need for lambda because .isnull works on the entire DataFrame and returns a Series with the count of null values for each column.
    'duplicated_%': (df.apply(lambda c: c.duplicated().sum()) / len(df) * 100).round(2) # duplicated() is series method, so we need to apply it to each column using apply() and then sum the boolean values to get the count of duplicated values for each column. Then we divide by the total number of rows and multiply by 100 to get the percentage of duplicated values in each column, rounded to 2 decimal places.
})
display(null_and_duplicated)

print(f"Missing values in 'medical_specialty': {df['medical_specialty'].isna().sum()}") #checking for missing values 
print(f"Missing values in 'transcription': {df['transcription'].isna().sum()}")

thresholds = [20, 50, 100, 200, 500] #transcription char length checking

threshold_summary = pd.DataFrame({
    'threshold': thresholds,
    'count': [( df['transcription'].str.len() < t).sum() for t in thresholds]}) #loop each threshhold and count len of transcriptions

display(threshold_summary)



Rows: 4999 Columns: 6


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


,null_%,duplicated_%
description,0.00,53.03
medical_specialty,0.00,99.20
sample_name,0.00,52.45
transcription,0.66,52.83
keywords,21.36,22.98


Missing values in 'medical_specialty': 0
Missing values in 'transcription': 33


,threshold,count
0,20,14
1,50,23
2,100,45
3,200,64
4,500,132


## Key Findings

After we take a look at initial data, we have table for percentage of null values in columns, and duplicated percentage in those columns medical specialty has 0 null value and 99% duplicated values and 

In [345]:
specialty_summary = ( 
    df.groupby('medical_specialty')['transcription'] #groupby medical specialty and count number of transcription
    .count()
    .reset_index()
    .rename(columns={'transcription': 'transcription_count'}) #rename
    .sort_values('transcription_count', ascending=False) #sorting 
    .assign(percentage=lambda x: (x['transcription_count'] / len(df) * 100
).round(2)))


display(specialty_summary)



,medical_specialty,transcription_count,percentage
38,Surgery,1088,21.76
5,Consult - History and Phy.,516,10.32
3,Cardiovascular / Pulmonary,371,7.42
27,Orthopedic,355,7.10
33,Radiology,273,5.46
15,General Medicine,259,5.18
14,Gastroenterology,224,4.48
22,Neurology,223,4.46
35,SOAP / Chart / Progress Notes,166,3.32
39,Urology,156,3.12


phase1.1 removing nulls, droping exact duplicates, and short notes under 20char

In [332]:
print(df.shape[0])
df = df.dropna(subset=['medical_specialty','transcription']) #to drop rows with missing values in the 'medical_specialty' column and transcription column
print(df.shape[0])

df['medical_specialty'] = df['medical_specialty'].str.strip() #to remove leading and trailing whitespace from the 'medical_specialty' column
df['transcription'] = df['transcription'].str.strip() #to remove leading and trailing whitespace from the 'transcription' column

df = df.drop_duplicates(subset=['transcription', 'medical_specialty']) #to drop duplicate rows based on the 'transcription' column, if they had exact same notes
print(df.shape[0])

df = df[df['transcription'].str.len() >= 20] #to filter the DataFrame to only include rows where the length of the 'transcription' column is at least 20 characters
print(df.shape[0]) #to check the shape of the DataFrame after filtering out short transcriptions


4999
4966
4964
4951


phase 1.1 feature extraction

In [ ]:
df['char_length'] = df['transcription'].str.len() #to create a new column 'char_length' that contains the length of the text in the 'transcription' column
df['word_count'] = df['transcription'].str.split().apply(len) #to create a new column 'word_count' that contains the number, splitting by whitespace
df['sentence_count'] = df['transcription'].apply(lambda note: len(sent_tokenize(note))) #to create a new column 'sentence_count' that contains the number of sentences in the 'transcription' column, we could use a regular function as well
df.head(3)


1.1 visuals

In [ ]:
plt.figure(figsize=(10, 8))
df['medical_specialty'].value_counts().plot(kind='barh')
plt.title('Number of notes per medical specialty')
plt.xlabel('Number of notes')
plt.ylabel('Specialty')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='word_count', bins=50)
plt.title('Distribution of note length (words)')
plt.xlabel('Words per note')
plt.ylabel('Number of notes')
plt.show()

In [ ]:
# keep only the 10 most common specialties so the plot stays readable
top_specialties = df['medical_specialty'].value_counts().head(10).index
subset = df[df['medical_specialty'].isin(top_specialties)]

plt.figure(figsize=(12, 7))
sns.boxplot(data=subset, x='word_count', y='medical_specialty')
plt.title('Note length by specialty (top 10)')
plt.xlabel('Words per note')
plt.ylabel('Specialty')
plt.tight_layout()
plt.show()

1.2 NLP analysis

In [ ]:
#cleaning the text data for vectorization and ML work
stop = set(stopwords.words('english'))

def clean(text):
    text = text.lower()                          # 1. lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)        # 2. remove punctuation/digits
    words = text.split()                         # 3. tokenize
    words = [w for w in words if w not in stop]  # 4. drop stopwords, using a list comprehension to filter out stopwords
    return ' '.join(words)                        # 5. rejoin

df['clean_text'] = df['transcription'].apply(clean)

df.head(1)

In [ ]:
all_words = ' '.join(df['clean_text']).split() #create a list of all words in the 'clean_text' column by joininn and splitting the text returning a list of words to counter work
top20 = Counter(all_words).most_common(20)
print("Top 20 most common words:", top20)

In [ ]:
print("total rows:", len(df))
print("unique transcriptions:", df['transcription'].nunique())
dup = df['transcription'].duplicated().sum()
print("duplicate transcription rows:", dup)

In [ ]:
for specialty in df['medical_specialty'].value_counts().head(5).index:
    notes = df[df['medical_specialty'] == specialty]['clean_text']
    words = ' '.join(notes).split()
    top = Counter(words).most_common(20)
    print(f"\n=== {specialty} ===")
    print(top) 

In [ ]:
df.to_csv('../data/processed/cleaned_medical_notes.csv', index=False) #to save the cleaned DataFrame to a new CSV file in the 'processed' directory, without including the index column

In [ ]:
words = ' '.join(df['clean_text']).split()      # same flat word list as before
bigrams = Counter(ngrams(words, 2)).most_common(20)
trigrams = Counter(ngrams(words, 3)).most_common(20)
print("Bigrams:", bigrams)
print("Trigrams:", trigrams)



phase 2.1, chi square test for whole text without any speciality

In [ ]:
# 1. build the document-term matrix from cleaned text
vectorizer = CountVectorizer(max_features=5000, stop_words =  ['x', 'xx', 'xxx', 'dr', 'md', 'mrs', 'dear', 'abc', 'xyz', 'mmddyyyy', 'yyyy', 'dd', 'yearold']) #to create a CountVectorizer object with a maximum of 5000 features, and remove some noise words like de identified words and common words that are not informative for specialty classification
X = vectorizer.fit_transform(df['clean_text']) #clean text notes for chi-square analysis 1part of matrix
y = df['medical_specialty'] #other part of patrix, the labels for chi-square analysis

# 2. run chi-square: score every word against the specialty labels
chi2_scores, p_values = chi2(X, y)

# 3. pair each word with its score
feature_names = vectorizer.get_feature_names_out()
scores = pd.DataFrame({'word': feature_names, 'chi2': chi2_scores})

# 4. the most "specialty-bonded" words overall
scores.sort_values('chi2', ascending=False).head(20)



phase2.1 chi square test for speciality based

In [ ]:
# specialties to profile — use exact labels from your data
specialties = df['medical_specialty'].value_counts().index

for target in specialties:
    # binary target: is this note the target specialty, yes/no?
    y_binary = (df['medical_specialty'] == target)

    # same X, same test — only y changed
    chi2_scores, p_values = chi2(X, y_binary)

    # pair words with scores, take the top 10 for this specialty
    scores = pd.DataFrame({
        'word': vectorizer.get_feature_names_out(),
        'chi2': chi2_scores
    }).sort_values('chi2', ascending=False).head(50)

    print(f"\n=== {target} ===")
    print(scores['word'].tolist())

phase 2.2, n gram analysis

In [ ]:
bigram_measures = BigramAssocMeasures()

# tokenize all cleaned text into one word list
words = ' '.join(df['clean_text']).split()

# build the finder (tallies co-occurrences + individual frequencies)
finder = BigramCollocationFinder.from_words(words)

# only keep pairs seen at least 5 times — PMI is unstable for rare pairs
finder.apply_freq_filter(5)

# score by PMI, take the top 20
top_pmi = finder.nbest(bigram_measures.pmi, 20)
top_pmi

In [ ]:
specialties = df['medical_specialty'].value_counts().index

for target in specialties:
    # this specialty's cleaned notes only
    notes = df[df['medical_specialty'] == target]['clean_text']
    words = ' '.join(notes).split()

    finder = BigramCollocationFinder.from_words(words)
    finder.apply_freq_filter(3)   # lower threshold — each specialty has fewer words

    top = finder.nbest(bigram_measures.pmi, 15)
    print(f"\n=== {target} ===")
    print(top)

Phase 2.3, TF-IDF and importance

In [ ]:
custom_noise = ['x', 'xx', 'xxx', 'dr', 'md', 'mrs', 'dear', 'abc', 'xyz', 'mmddyyyy', 'yyyy', 'dd', 'yearold'] #to create a list of noise words to remove from the text data during vectorization, these words are identified as de identified words and common words that are not informative for specialty classification
tfidf = TfidfVectorizer(
    max_features=10000,        # cap vocabulary size
    ngram_range=(1, 3),        # unigrams + bigrams + trigrams
    stop_words=custom_noise    # your artifact list
)

X_tfidf = tfidf.fit_transform(df['clean_text'])
print(X_tfidf.shape)   # (n_notes, n_features)

In [ ]:
# pair each word with its IDF (lower = more common, higher = rarer/more distinctive)
idf = pd.DataFrame({
    'word': tfidf.get_feature_names_out(),
    'idf': tfidf.idf_
})

print("LOWEST IDF (most common, treated as noise):")
print(idf.sort_values('idf').head(15))

print("\nHIGHEST IDF (rarest, most distinctive):")
print(idf.sort_values('idf', ascending=False).head(15))